# 05 組み合わせ・検知限界・最終評価 — どこまで使えるかを確かめる

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: 手法を組み合わせ、「どこまで小さい異常まで見つけられるか（検知限界）」を調べる回です。
> - **たとえ話**: 虫眼鏡で、どこまで小さい傷が見えるかを調べる。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-6（注入と検知限界）・2-7（データの分け方）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

## ゴール
- 異常の種類ごとに得意な手法を**組み合わせ**て、1つの検知器にする。
- 劣化が**何%低下から**見つかるか（**検知限界**）、影が**どれだけ弱いと**見逃すかを調べる。
- 「注入した異常が、簡単すぎないか」を疑い、**結果の限界**を理解する。
- 最終評価（testは1回だけ）の考え方を知る。

## 前のNotebookとのつながり
01〜04 で、手法ごとに得意・不得意が違うことが分かりました。
- 日次の残差（02）: 停電・影に強い。固着は弱い。劣化は遅い。
- 7日平均の残差（02）: 劣化に強いが、異常直後に警報が居残る（余韻）。
- 固着ルール（01）: 固着だけに強い。

ここでは、これらを**どれか1つでも警報を出せば警報（OR）**で組み合わせます。

## やること
1. 3つの構成を比べて、最終構成を決める
2. 劣化の検知限界を調べる
3. 影の強さを変えて、感度分析をする
4. 予行演習（val）で最終構成を評価する

## キーワード
- **検知限界**: これより小さい異常は見つけられない、という境目。
- **感度分析**: 異常の大きさなどの条件を変えて、結果がどう変わるかを調べること。
- **最終評価**: 手法・しきい値をすべて決めたあと、**1回だけ**未使用のデータ（test）で評価すること。
- **予行演習**: 最終評価の手順を、使い捨てでない別のデータ（val の未使用の系列）で試すこと。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import run_phase5
import final_eval

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
summ, res = run_phase5.compare(verbose=False)
summ

**何をしているか**: 3つの構成を、val の10系列で比べます。しきい値はすべて train（正常のみ）で決めます。
- 日次のみ: 02 の日次の残差（正常日の警報率3%）
- 日次＋固着: ↑に、同じ値が続く固着ルールを足す
- 劣化重視: 日次の残差（2%）＋7日平均の残差（1%）＋固着ルール

**出力の見方**:
- **日次のみ → 日次＋固着**: 誤警報（`fa`）は月0.85→0.86回とほぼ同じで、適合率は0.78→0.81に上がります。固着ルールは、正常日にほとんど警報を出さない（誤警報の増加がほぼゼロ）ので、**足すだけ得**です。
- **劣化重視**: 7日平均の残差を足すと、誤警報が月4.48回に増えます。ただし、`fa_grace7`（異常直後7日の正常日を除く）は0.74回です。つまり**余韻**による見かけの増加です。適合率は0.49まで下がります。
- 最終構成は、**誤警報が少なく適合率が高い「日次＋固着」**としました。劣化重視は、「劣化の早期発見を優先し、余韻の警報は許容する」場合の選択肢です。

In [ ]:
print("== 検知率（イベント単位） ==")
print(res.pivot(index="config", columns="type", values="recall").loc[list(run_phase5.CONFIGS)])
print("\n== 日単位の検知率 ==")
print(res.pivot(index="config", columns="type", values="day_recall").loc[list(run_phase5.CONFIGS)])
print("\n== 検知までの日数 ==")
print(res.pivot(index="config", columns="type", values="delay").loc[list(run_phase5.CONFIGS)])

**出力の見方**:
- **日次＋固着**は、4種類すべてで、イベント単位1.0です。停電・影・固着は日単位でも1.0です。
- 劣化だけは、日単位0.29（検知まで約7日）です。7日平均を足すと0.51に上がります。それでも約半分の日は見逃します。
- つまり、**劣化は「徐々に進む」ので、見つかるのは悪化が進んでから**です。次のセルで、どの低下率から見つかるかを調べます。

In [ ]:
lim, base = run_phase5.degrade_limit(verbose=False)
print("劣化以外の日の警報率:", base)
print(lim.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3.8))
xs = np.arange(len(lim))
ax.plot(xs, lim["daily_rate"], marker="o", label="日次の残差")
ax.plot(xs, lim["avg7_rate"], marker="s", label="7日平均の残差")
ax.axhline(base["daily"], color="gray", ls="--", lw=1, label="正常日の警報率（目安）")
ax.set_xticks(xs); ax.set_xticklabels(lim["bin"])
ax.set(xlabel="その日の低下率", ylabel="警報が出た割合", title="劣化の検知限界", ylim=(0, 1.05)); ax.legend()
fig.tight_layout()

**何をしているか**: 劣化だけを注入した40系列（最終的な低下率20%）で、**日ごとの低下率**（本来の値より何%少ないか）ごとに、警報が出た割合を調べます。劣化以外の日の警報率（約2%）が、「当てずっぽうで警報が出る割合」の目安です。

**出力の見方**:
- 低下率が**2〜4%**: 警報率は日次で0.08、7日平均で0.00。**ほぼ見つかりません**。
- **4〜6%**: 日次0.16、7日平均0.06。まだ偶然に近い水準です。
- **6〜8%**: 日次0.20、7日平均0.29。見つかり始めます。
- **8〜10%**: 日次0.44、7日平均0.78。7日平均が日次を大きく上回ります。
- **10〜15%**: 日次0.67、7日平均0.99。**10%を超えると、7日平均でほぼ確実に見つかります**。
- 結論: この設定（発電量のばらつき約3%＋日射計測誤差約4%）では、劣化の**検知限界は、およそ6〜8%の低下**です。7日平均を使うと、約8〜10%で8割が見つかります。
- 低下が5%以下の劣化は、**この方法では実質的に見つけられません**。ノイズの大きさが、検知限界を決めています。

In [ ]:
shade = run_phase5.shade_sensitivity(verbose=False)
shade

**何をしているか**: 影の強さ（発電量が本来の何%減るか）を変えて、影だけを3回注入した40系列で、日次＋固着の検知器を評価します。

**出力の見方**:
- **20%以上の低下**: イベント単位も日単位も1.0（0.99〜1.0）。これまでの実験（00 で作った影は2〜4割低下）は、**検知しやすい強さ**でした。
- **15%**: 日単位0.90。**10%**: 0.60。**5%**: 0.19（イベント単位0.64）。
- つまり、**影が弱い（10%以下）と、見逃しが増えます**。これまでの「影は日単位1.0」は、**注入した影が強かった**ためです。**実際の影は、もっと弱いかもしれません**。
- このように、**注入した異常の大きさが結果を左右します**。検知率の数字だけを見て「完璧に近い」と思わず、異常の大きさの前提を確認する必要があります。

In [ ]:
res_v, s_v, order = final_eval.evaluate("val")   # 予行演習: val の index=11〜20（開発では使っていない系列）
print("予行演習（val index=11〜20）")
print(s_v)
print(res_v.pivot(index="detector", columns="type", values="recall").loc[order])
print(res_v.pivot(index="detector", columns="type", values="day_recall").loc[order])

**何をしているか**: 最終評価と**同じ手順**を、val の未使用の系列（index=11〜20）で試す予行演習です。構成は変えません。

**出力の見方**:
- 最終構成の誤警報は月1.10回、適合率0.76です。開発に使った系列（index=1〜10: 0.86回・0.81）と**近い値**です。
- 参照として入れた素朴ルール（3kWh未満）は、誤警報が月4.33回で、適合率0.26、影・固着はほとんど見つけられません。
- 最終評価（test）は、`python src/final_eval.py --split test` で**1回だけ**実行できます。実行後は `outputs/final/TEST_USED.json` が作られ、2回目は拒否されます。
  **テストの結果を見て、構成やしきい値を変えない**のが約束です。結果は [REPORT.md](../REPORT.md) にあります。

## 試してみよう（演習）
1. **劣化の深さを変える**: `src/run_phase5.py` の `degrade_depth=(0.20, 0.20)` を `(0.10, 0.10)` に変えて、`degrade_limit()` を実行してください。
   見るポイントは、検知される低下率の境目が動くかどうかです。分かるのは、境目は最終的な深さではなく、**その日の低下率**で決まるということです。
2. **ノイズを増やす**: `src/anomaly_synth/generator.py` の日射の計測誤差 `0.04` を `0.08` に変えて、`python src/make_data.py` と `degrade_limit()` を実行してください。
   見るポイントは、検知限界（警報率が上がり始める低下率）の変化です。分かるのは、検知限界がノイズの大きさで決まることです。実験後は、元の値 `0.04` に戻してください。
3. **誤警報の目標を下げる**: `CONFIGS` の `0.03` を `0.01` に変えて、`compare()` を実行してください。見るポイントは、誤警報が減る代わりに、劣化の日単位の検知率がどう変わるかです。

## まとめ
- **異常の種類ごとに得意な手法を組み合わせる**と、停電・影・固着・劣化のすべてをイベント単位で見つけられた（日次＋固着）。
- ただし劣化は、**6〜8%を超える低下**から見つかり始める。5%以下は実質的に見つけられず、これはノイズの大きさで決まる。
- 影は、20%以上の低下なら確実だが、**10%以下では見逃しが増える**。検知率は、**注入した異常の大きさに左右される**。
- 最終評価は、構成を決めてから**1回だけ**行う。予行演習で手順を確認し、結果を見て構成を変えない。

次は Phase6（実データ）です。自宅の発電量に残差法を適用し、検知日を目視確認します。実データは手元だけで扱い、結果はコミットしません。